# NIH Chest X-ray - Explainability

This notebook evaluates the models. The resulting file is stored.

## Contents:
1. Environment Setup & Imports
2. Experiment Configuration & Reproducibility
3. Device Configuration
4. Dataset Loading
5. Model Loading
6. XAI Method Initialization
7. Inference & Sample Selection
8. Grad-CAM Visualization
9. Attention Rollout Visualization (ViT)
10. Failure Case Analysis
11. Failure Case Visualization
12. Summary of Explainability Results

## 1. Imports & Setup

This section imports visualization and explainability utilities used to interpret model predictions on chest X-ray images.

In [2]:
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

import config
from dataset import get_dataloaders
from models import build_model
from explainability import GradCAM, AttentionRollout, overlay_heatmap
from evaluate import collect_predictions
from utils import seed_everything

## 2. Experiment Setup

This section defines the experiment and ensures reproducibility for explainability analysis.

In [3]:
seed_everything()

EXPERIMENT_NAME = "exp_01_densenet_baseline"
config.set_experiment(EXPERIMENT_NAME)

print("Experiment:", config.EXPERIMENT_NAME)

Experiment: exp_01_densenet_baseline


## 3. Device Setup

This section selects the computation device for running inference and explainability methods.

In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

device

device(type='mps')

## 4. Load Dataset

This section loads the test dataset used for generating explanations.

In [5]:
train_loader, val_loader, test_loader = get_dataloaders()

print("Test samples:", len(test_loader.dataset))

[dataset] Loading metadata ...
[dataset] Subset mode: 1002 train+val, 202 test images
[dataset] Train/val patient overlap: 0
[dataset] Split sizes - train: 894, val: 108, test: 202
Test samples: 202


## 5. Load Trained Model

This section loads the trained model checkpoint for generating explanations.

In [6]:
model_name = "densenet121"

model = build_model(model_name, pretrained=False).to(device)

ckpt_path = config.CHECKPOINT_DIR / f"{model_name}_best.pt"
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

print("Loaded:", ckpt_path)

Loaded: /Users/vesco/Documents/Projects/xray-master/outputs/exp_01_densenet_baseline/checkpoints/densenet121_best.pt


## 6. Initialize XAI Method

Grad-CAM highlights spatial regions in the X-ray that most influenced the CNN prediction.
Attention Rollout aggregates transformer attention maps to identify influential image patches.

In [7]:
cam = GradCAM(model)
#rollout = AttentionRollout(model)

## 7. Run Inference + Select Samples

This section selects a subset of test samples for qualitative explainability analysis.

In [8]:
labels, probs = collect_predictions(model, test_loader, device)

print("Collected predictions:", labels.shape)

Collected predictions: (202, 15)


## 8. Visualize Grad-CAM (Correct Predictions)

This section visualizes Grad-CAM heatmaps for correctly predicted cases to validate whether the model focuses on clinically relevant regions.

In [9]:
config.XAI_DIR.mkdir(parents=True, exist_ok=True)

sample_count = 0
num_samples = 5

for batch in test_loader:
    images = batch["image"]
    filenames = batch["filename"]

    for i in range(images.size(0)):
        if sample_count >= num_samples:
            break

        img = images[i].unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(img)
            prob = torch.sigmoid(logits)[0]

        pred_class = int(prob.argmax())

        heatmap = cam.generate(img, pred_class)

        save_path = config.XAI_DIR / f"gradcam_{sample_count}.png"

        overlay_heatmap(
            image_tensor=images[i],
            heatmap=heatmap,
            title=f"Grad-CAM - {config.CLASS_NAMES[pred_class]}",
            save_path=str(save_path)
        )

        sample_count += 1

## 9. Visualize Attention Rollout (ViT)

This section visualizes transformer attention maps to understand patch-level reasoning in ViT models.

In [10]:
if model_name == "vit_b_16":

    sample_count = 0
    num_samples = 5

    for batch in test_loader:
        images = batch["image"]

        for i in range(images.size(0)):
            if sample_count >= num_samples:
                break

            img = images[i].unsqueeze(0).to(device)

            heatmap = rollout.generate(img)

            save_path = config.XAI_DIR / f"attention_{sample_count}.png"

            overlay_heatmap(
                image_tensor=images[i],
                heatmap=heatmap,
                title="Attention Rollout",
                save_path=str(save_path)
            )

            sample_count += 1

## 10. Failure Case Analysis

This section analyzes incorrectly predicted samples to identify systematic model weaknesses.

In [11]:
wrong_samples = []

for batch in test_loader:
    images = batch["image"]

    for i in range(images.size(0)):
        img = images[i].unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(img)
            prob = torch.sigmoid(logits)[0]

        pred = (prob > 0.5).int().cpu().numpy()
        true = batch["label"][i].numpy()

        if not np.array_equal(pred, true):
            wrong_samples.append((images[i], pred, true))

        if len(wrong_samples) >= 5:
            break

print("Collected failure cases:", len(wrong_samples))

Collected failure cases: 11


## 11. Visualize Failure Cases

This section visualizes misclassified samples to highlight model limitations and dataset challenges.

In [12]:
for i, (img, pred, true) in enumerate(wrong_samples):

    save_path = config.XAI_DIR / f"failure_{i}.png"

    overlay_heatmap(
        image_tensor=img,
        heatmap=np.zeros((img.shape[1], img.shape[2])),  # no heatmap for baseline view
        title=f"Pred: {pred} | True: {true}",
        save_path=str(save_path)
    )

## 12. Summary of Explainability Results

This section summarizes qualitative findings from Grad-CAM and Attention Rollout visualizations.

In [13]:
print("Explainability complete.")
print("Check outputs/xai/ for visual results.")

Explainability complete.
Check outputs/xai/ for visual results.
